# telefon + sise Güçlendirme (Colab)
9-sınıf detektörde **telefon (mAP 0.33) + sise (0.29) ZAYIF** — sebep: COCO'nun
generic phone/bottle'ı FTR in-cabin domaine uymuyor + dengesizlik.
**Fix:** abnormal_behaviour'ın sürücü-içi Phone→telefon + Drinking→sise verisini ekle,
yeniden eğit. repo/token GEREKMEZ (self-contained). Runtime → T4 GPU.


## 1) GPU + Drive


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip -q install ultralytics
import torch
assert torch.cuda.is_available(), 'GPU KAPALI! Runtime -> T4 GPU'
print('GPU:', torch.cuda.get_device_name(0))


## 2) augmented dataset'i geri yükle


In [ ]:
import os
!unzip -qo /content/drive/MyDrive/rapid_response_9class_aug.zip -d /content/datasets/
print('dataset:', os.path.exists('/content/datasets/rapid_response/images/train'),
      '| train:', len(os.listdir('/content/datasets/rapid_response/images/train')))


## 3) abnormal_behaviour ekle (in-cabin telefon+sise, train+val'e böl)
abnormal idx: 0 Cigarette,1 Drinking,2 Eating,3 Phone,4 Seatbelt ->
bizim: 5 sigara, 6 sise, (Eating atlanir), 4 telefon, 7 emniyet_kemeri


In [ ]:
import os, glob, shutil, random
if not os.path.isdir('/content/abnormal_local'):
    print('abnormal kopyalaniyor (~10 dk)...')
    !cp -r "/content/drive/MyDrive/dataset/sofor_eylemi/abnormal_behaviour" /content/abnormal_local
remap={0:5, 1:6, 3:4, 4:7}
sI='/content/abnormal_local/train/images'; sL='/content/abnormal_local/train/labels'
D='/content/datasets/rapid_response'
files=glob.glob(sL+'/*.txt'); random.seed(0); random.shuffle(files)
nval=max(1,int(len(files)*0.15)); added={'train':0,'val':0}
for i,lf in enumerate(files):
    sp='val' if i<nval else 'train'
    b=os.path.splitext(os.path.basename(lf))[0]
    lines=[f"{remap[int(p.split()[0])]} {' '.join(p.split()[1:])}"
           for p in open(lf) if p.split() and int(p.split()[0]) in remap]
    if not lines: continue
    img=next((sI+'/'+b+e for e in ('.jpg','.jpeg','.png') if os.path.exists(sI+'/'+b+e)), None)
    if not img: continue
    shutil.copy(img, f"{D}/images/{sp}/abn_{b}{os.path.splitext(img)[1]}")
    open(f"{D}/labels/{sp}/abn_{b}.txt",'w').write('\n'.join(lines)+'\n')
    added[sp]+=1
for c in glob.glob(D+'/labels/*/*.cache'): os.remove(c)
print('eklendi -> train:', added['train'], 'val:', added['val'])


## 3b) geliştirilmiş dataset'i KAYDET (reset'e karşi)


In [ ]:
%cd /content/datasets
!zip -qr /content/drive/MyDrive/rapid_response_9class_v2.zip rapid_response
print('v2 dataset kaydedildi')


## 4) eğit (v2)


In [ ]:
import yaml
yaml.safe_dump({'path':'/content/datasets/rapid_response','train':'images/train','val':'images/val','test':'images/test',
 'nc':9,'names':['arac','plaka','teknocan','bilgisayar','telefon','sigara','sise','emniyet_kemeri','kisi']},
 open('/content/data.yaml','w'))
from ultralytics import YOLO
YOLO('yolo11s.pt').train(data='/content/data.yaml', epochs=50, imgsz=640, batch=16, device=0,
                         project='/content/runs', name='rr2')


## 5) model kaydet


In [ ]:
!cp /content/runs/rr2/weights/best.pt /content/drive/MyDrive/best_model_9class_v2.pt
!cp -r /content/runs/rr2 /content/drive/MyDrive/yolo_sonuc_9class_v2
print('v2 model kaydedildi')
